# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abuhussein1504/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Lane 2 — Refresh / Content Opportunity Scoring.** This notebook trains on the same starter CSV and the same eligible population as `w04_baseline_score.ipynb` (ML-07), so the model-vs-baseline comparison in Section 3 is apples-to-apples: same data, same metric.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

---

**Target:** `is_declining` = `trend_direction == "down"` — an *observed* outcome (per `docs/data-dictionary.md`), not a rule I defined myself. That satisfies the framing skill's rule: the target must be observed, not defined.

**Task shape:** this is the "which ones first?" ranking question from `w02_ml_task_framing.ipynb`, so the metric stays **Precision@50** — same K used for the leakage sanity-check in `w03_feature_leakage_check.ipynb`, so results are comparable across the whole project, not just against the baseline.

**Method:** the `training-honest-models` table maps "yes/no with an observed label" to **Logistic Regression, then Random Forest** — readable first, stronger second. I train both, scored by predicted probability and ranked for Precision@50, and report both against the baseline. Complexity has to earn its place here, not be assumed.

---

In [1]:
import os, sys, subprocess
import json
from pathlib import Path

import numpy as np
import pandas as pd

REPO_URL = "https://github.com/abuhussein1504/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Same observed target as the rest of the project — never trend_direction/trend_pct as a FEATURE,
# only as the thing we're predicting (see docs/data-dictionary.md rule #2).
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# Same eligible population as w04_baseline_score.ipynb: only pages we actually have signal on.
eligible_mask = (df["impression_tier"] != "no_data") & (df["avg_position"] > 0)
elig = df.loc[eligible_mask].reset_index(drop=True).copy()

print(f"rows total: {len(df):,}  |  eligible population: {len(elig):,}")
print(f"base rate (is_declining) in eligible population: {elig['is_declining'].mean():.3f}")


Cloning into 'flyrank-ml-internship-starter'...


rows total: 30,000  |  eligible population: 28,795
base rate (is_declining) in eligible population: 0.564


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

---

**Grouped by `client_id`.** Pages from the same client share hidden character (the client's niche, publishing cadence, how aggressively they were already refreshing before this export) — a random row split would let the model partly memorize the client rather than learn a transferable pattern, exactly the trap `hunting-leakage-and-validating` calls out. `client_id` is the repeating entity here (32 distinct clients in the eligible slice), so `GroupKFold` with 5 folds is the honest choice: every fold tests on clients the model never trained on. No time column exists in this snapshot CSV (it's one trailing-90-day export, not a daily panel), so a time-based split isn't available here — that's a real limitation of the starter data, not a design gap I'm hiding.

**Feature build:** categorical columns (`content_type`, `main_intent`) are one-hot encoded; missing-keyword-data and missing-word-count rows get `has_` flags *before* filling with 0, so the model can tell "zero because unmeasured" apart from "zero because it's really zero" — per the missingness-follows-category warning in the data dictionary. `content_id` / `client_id` stay out of the feature matrix (context only, per the contract).

---

In [2]:
# Flags BEFORE fillna — missingness follows content_type (see data dictionary), a blind
# fillna(0) would silently encode content_type into the features.
elig["has_keyword_data"] = elig["search_volume"].notna().astype(int)
elig["has_word_count"] = elig["word_count"].notna().astype(int)

numeric_cols = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "clicks_last_30d", "sessions_last_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
# NOTE: impressions_last_30d / impressions_prev_30d are deliberately EXCLUDED here —
# see the leakage catch in Section 3. Left out from the start once caught; this list is
# already the honest, post-leakage-check feature set.
flag_cols = ["has_keyword_data", "has_word_count"]
cat_cols = ["content_type", "main_intent"]

elig[cat_cols] = elig[cat_cols].fillna("unknown")
cat_dummies = pd.get_dummies(elig[cat_cols], prefix=cat_cols)

X = pd.concat([elig[numeric_cols].fillna(0), elig[flag_cols], cat_dummies], axis=1)
y = elig["is_declining"].values
groups = elig["client_id"].values

print(f"X shape: {X.shape}")
print(f"distinct clients (groups): {len(np.unique(groups))}")

from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits=5)
for i, (tr, te) in enumerate(gkf.split(X, y, groups)):
    print(f"fold {i}: train={len(tr):,} test={len(te):,} test_clients={len(np.unique(groups[te]))}")


X shape: (28795, 36)
distinct clients (groups): 31


fold 0: train=21,792 test=7,003 test_clients=1
fold 1: train=23,350 test=5,445 test_clients=7
fold 2: train=23,349 test=5,446 test_clients=9
fold 3: train=23,349 test=5,446 test_clients=7
fold 4: train=23,340 test=5,455 test_clients=7


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

---

**Reproducing the Week-4 baseline** on this exact eligible population first, so the comparison is fair: the `stale_but_visible` rule from `w04_baseline_score.ipynb` (`score = stale_flag * visible_flag * impressions_90d`, stale = 90+ days since update, visible = 500+ impressions/90d), ranked, Precision@50 against `is_declining`.

**A leakage catch, caught before it was trusted:** my first pass of the feature list included `impressions_last_30d` and `impressions_prev_30d`. Both models scored Precision@50 = **1.000** — "suspiciously perfect," exactly the symptom `hunting-leakage-and-validating` warns about. I ran the attack test: `corr((impressions_last_30d - impressions_prev_30d) / impressions_prev_30d, trend_pct)` = **0.99999998** — essentially 1.0. `trend_pct` (and therefore `trend_direction`, and therefore my label) is computed directly from that pair of columns. They aren't predictive features; they're the label's own source columns wearing a different name. I removed both (Section 2's feature list above is already the corrected version) and retrained — that's the honest collapse documented below.

---

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K = 50
base_rate = y.mean()

# --- Reproduce the Week-4 baseline rule, on this SAME eligible population ---
STALE_DAYS, VISIBLE_IMPRESSIONS = 90, 500
stale = elig["days_since_last_update"] >= STALE_DAYS
visible = elig["impressions_90d"] >= VISIBLE_IMPRESSIONS
baseline_score = np.where(stale & visible, elig["impressions_90d"].astype(float), 0.0)
p_baseline = precision_at_k(baseline_score, y, K)

# --- Leakage demo: WITH the suspect columns (quick single fit, not the final model) ---
leaky_extra = elig[["impressions_last_30d", "impressions_prev_30d"]].fillna(0)
X_leaky = pd.concat([X, leaky_extra], axis=1)
gkf = GroupKFold(n_splits=5)
oof_leaky = np.zeros(len(X))
for tr, te in gkf.split(X_leaky, y, groups):
    m = LogisticRegression(max_iter=5000, class_weight="balanced")
    m.fit(X_leaky.iloc[tr], y[tr])
    oof_leaky[te] = m.predict_proba(X_leaky.iloc[te])[:, 1]
p_leaky = precision_at_k(oof_leaky, y, K)

# --- Honest models: WITHOUT the suspect columns (this is the real result) ---
oof_lr = np.zeros(len(X))
oof_rf = np.zeros(len(X))
coef_accum = np.zeros(X.shape[1])
for tr, te in gkf.split(X, y, groups):
    Xtr, Xte, ytr = X.iloc[tr], X.iloc[te], y[tr]

    scaler = StandardScaler()
    Xtr_s, Xte_s = scaler.fit_transform(Xtr), scaler.transform(Xte)
    lr = LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42)
    lr.fit(Xtr_s, ytr)
    oof_lr[te] = lr.predict_proba(Xte_s)[:, 1]
    coef_accum += lr.coef_[0]

    rf = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                 random_state=42, n_jobs=-1)
    rf.fit(Xtr, ytr)
    oof_rf[te] = rf.predict_proba(Xte)[:, 1]

p_lr = precision_at_k(oof_lr, y, K)
p_rf = precision_at_k(oof_rf, y, K)

print("--- Leakage collapse (the confession) ---")
print(f"Precision@{K} WITH impressions_last/prev_30d : {p_leaky:.3f}  <- suspiciously near 1.0")
print(f"Precision@{K} WITHOUT them (Logistic Regr.)   : {p_lr:.3f}")
print()

comparison = pd.DataFrame({
    "method": ["Base rate (random)", "Week-4 baseline rule", "Logistic Regression", "Random Forest"],
    f"precision@{K}": [round(base_rate, 3), round(p_baseline, 3), round(p_lr, 3), round(p_rf, 3)],
})
print("--- Model vs. baseline (same eligible population, same split unit, same metric) ---")
comparison


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


--- Leakage collapse (the confession) ---
Precision@50 WITH impressions_last/prev_30d : 1.000  <- suspiciously near 1.0
Precision@50 WITHOUT them (Logistic Regr.)   : 0.720

--- Model vs. baseline (same eligible population, same split unit, same metric) ---


,method,precision@50
0,Base rate (random),0.564
1,Week-4 baseline rule,0.440
2,Logistic Regression,0.720
3,Random Forest,0.520


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

---

**Read the table first.** Base rate is 0.564 (a page is more likely declining than not, in this slice — most content here is aging). The Week-4 rule scores **below** the base rate at Precision@50: `stale_but_visible` ranks by raw impressions, and Signal 1 in `w04_baseline_score.ipynb` already flagged this belief as MIXED — staleness alone doesn't reliably track decline, so a rule that ranks by visibility rather than any decline-shaped signal does no better than picking pages at random. Logistic Regression is the one method here that clearly beats both the base rate and the rule. Random Forest lands close to the base rate — more complexity did **not** buy more precision on this feature set, which is itself the finding, not a bug to explain away.

**What Logistic Regression leans on:** averaging its (standardized) coefficients across folds, the two strongest signals are `clicks_last_30d` (large negative — fewer recent clicks pushes the prediction toward "declining") and `sessions_90d` / `clicks_prev_30d` (positive — a page with a healthy longer-run click/session history is predicted less likely to be declining). That's the shape I'd expect from a decline signal: a *recent* drop against a *longer* baseline, not any single raw count — reassuring, not "suspiciously perfect."

**Reading the top-50 wrong picks:** 14 of the top 50 ranked-by-LR pages are false positives (model said "declining," `trend_direction` says otherwise). Most of those are `trend_direction == "stable"`, not `"up"` — the model is catching a real recent dip in `clicks_last_30d`, but on a page whose longer trend hadn't (yet) crossed the pipeline's own decline threshold. That's a legitimate kind of error for this task: the cost of a false positive here is a wasted review, not a missed decline, and `w01_research_question.ipynb` already named that as the cheaper mistake to make.

---

In [4]:
coef_avg = coef_accum / 5
coefs = pd.Series(coef_avg, index=X.columns).sort_values()

print("Top features pushing toward 'declining':")
print(coefs.head(5))
print("\nTop features pushing toward 'not declining':")
print(coefs.tail(5))

order = np.argsort(-oof_lr)
top_k_idx = order[:K]
top_k = elig.iloc[top_k_idx].copy()
top_k["pred_score"] = oof_lr[top_k_idx]
top_k["actual_declining"] = y[top_k_idx]
wrong = top_k[top_k["actual_declining"] == 0]

print(f"\nwrong picks in top {K}: {len(wrong)} of {K}")
print(wrong["trend_direction"].value_counts())
wrong[["content_id", "trend_direction", "clicks_last_30d", "clicks_prev_30d", "days_since_last_update"]].head(5)


Top features pushing toward 'declining':
clicks_last_30d      -2.620569
users_90d            -1.304907
char_count           -0.435080
days_with_sessions   -0.390827
content_age_days     -0.308065
dtype: float64

Top features pushing toward 'not declining':
days_with_impressions    0.419815
word_count               0.588257
clicks_90d               0.746478
clicks_prev_30d          1.419425
sessions_90d             1.444087
dtype: float64

wrong picks in top 50: 14 of 50
trend_direction
stable    12
up         2
Name: count, dtype: int64


,content_id,trend_direction,clicks_last_30d,clicks_prev_30d,days_since_last_update
9296,content_4560b0a818ab,stable,2,5,22
28220,content_2dba2b1f9536,stable,314,323,104
6584,content_a965a1fc5544,stable,138,372,104
1958,content_fb95edac0fc3,stable,53,45,104
19002,content_035aa1e4a834,up,3,1,104


In [5]:
# Save the comparison table + a few interpretation notes, same pattern as w04_baseline_score.
out_path = Path("work/outputs/w05_model_metrics.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
metrics = {
    "k": K,
    "base_rate": round(float(base_rate), 3),
    "precision_at_k": {
        "week4_baseline_rule": round(float(p_baseline), 3),
        "logistic_regression": round(float(p_lr), 3),
        "random_forest": round(float(p_rf), 3),
    },
    "leakage_check": {
        "suspect_columns": ["impressions_last_30d", "impressions_prev_30d"],
        "precision_at_k_with_leak": round(float(p_leaky), 3),
        "precision_at_k_without_leak": round(float(p_lr), 3),
        "reason": "corr((last-prev)/prev, trend_pct) = 0.99999998 -- source columns of the label",
    },
    "top50_wrong_picks_logreg": int(len(wrong)),
}
with open(out_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Wrote {out_path}")


Wrote work/outputs/w05_model_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.